In [1]:
import os
import json
import base64
import numpy as np
import pandas as pd
import geopandas as gpd
from typing import Optional
from pyiceberg.expressions import And, GreaterThanOrEqual, LessThanOrEqual
from pyiceberg.catalog import load_catalog

In [2]:
import numpy as np
import geopandas as gpd
from scipy.spatial import cKDTree

def search_spatial_candidates(
    reference_gdf: gpd.GeoDataFrame,
    compared_gdf: gpd.GeoDataFrame,
    k: int = 100,
    max_dist: float = 1000, 
    id_col: str = "id",
    crs_for_distance: int = 3857,
):
    """
    Attach k nearest compared POI ids & distances to reference_gdf.

    Returns
    -------
    GeoDataFrame with two new columns:
    - cand_ids   : list of compared ids
    - cand_dist_m: list of distances (meters)
    """

    ref_proj = reference_gdf.to_crs(crs_for_distance)
    cmp_proj = compared_gdf.to_crs(crs_for_distance)

    ref_xy = np.column_stack([ref_proj.geometry.x, ref_proj.geometry.y])
    cmp_xy = np.column_stack([cmp_proj.geometry.x, cmp_proj.geometry.y])

    tree = cKDTree(cmp_xy)
    k_eff = min(k, len(compared_gdf))

    dist, idx = tree.query(ref_xy, k=k_eff)

    if k_eff == 1:
        dist = dist.reshape(-1, 1)
        idx = idx.reshape(-1, 1)

    cmp_ids = compared_gdf[id_col].to_numpy()

    cand_ids = []
    cand_dists = []

    for row_idx, row_dist in zip(idx, dist):
        ids = []
        dists = []

        for j, d in zip(row_idx, row_dist):
            if d <= max_dist:
                ids.append(cmp_ids[j])
                dists.append(d)

        cand_ids.append(ids)
        cand_dists.append(dists)

    result = reference_gdf.copy()
    result["cand_ids"] = cand_ids
    result["cand_dist_m"] = cand_dists

    return result

In [3]:
FOOD_WORDS = {
    "RESTAURANT","RESTURANT","RESTARAUNT",
    "CAFE","CAFÉ","COFFEE","BAR","BISTRO","GRILL",
    "KITCHEN","DINER","EATERY","STEAKHOUSE",
    "SEAFOOD","BUFFET","BBQ","PIZZA","SUSHI","RAMEN",
    "NOODLE","NOODLES","BURGER","BURGERS","TACO","TACOS",
    "CHICKEN","WINGS","BAKERY","DELI","DELICATESSEN",
    "COURT","FOOD","EXPRESS","HOUSE","SHOP"
}

PLACE_WORDS = {
    "HALL","CENTER","CENTRE","PLAZA","MARKET","MALL",
    "GARDEN","GARDENS","PARK","SQUARE","TOWER","TOWERS",
    "STATION","TERMINAL","BUILDING","GALLERY","THEATER","SCHOOL","COURT","INN",
    "HOTEL","MOTEL","INN","SUITES","SUITE",
    "SPA","SALON","STUDIO","CENTER","CENTRE",
    "CLUB","LOUNGE","STATION","STOP"
}

LEGAL_WORDS = {
    "LLC","INC","CORP","CORPORATION","CO","COMPANY",
    "LTD","LIMITED","GROUP","HOLDINGS","OFFICE"
}

GRAMMAR = {
    "THE","OF","AT","AND","FOR","IN","ON","BY","WITH","TO","FROM"
}

NON_PRIMARY_TOKENS = FOOD_WORDS | PLACE_WORDS | LEGAL_WORDS | GRAMMAR

In [4]:
from rapidfuzz import process, fuzz
import pandas as pd
import re
import unicodedata


def clean_name(s):
    if not isinstance(s, str):
        return ""

    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c)) # 1. unicode normalize (remove accents)
    s = s.upper() # 2. uppercase
    s = re.sub(r"\([^)]*\)", "", s) 
    s = re.sub(r"\b'S\b", "", s) # new change
    s = re.sub(r"\bS\b", "", s) # new change
    s = s.encode("ascii", errors="ignore").decode() # 4. remove emoji / non ascii
    s = re.sub(r"[^\w\s]", " ", s) # 5. replace special chars with space
    s = re.sub(r"\s+", " ", s) # 6. collapse spaces

    return s.strip()

def extract_prinmary_str(name):

    tokens = name.split()
    core = [t for t in tokens if t not in NON_PRIMARY_TOKENS]
    if len(core) == 1 and len(core[0]) < 3:
        return name
    if core:
        return " ".join(core)
    return name

def match_by_name(
    reference_gdf: gpd.GeoDataFrame,
    compared_gdf: gpd.GeoDataFrame,
    re_name_col: str = "name",
    comp_name_col: str = "name",
    comp_id: str = "id",
    comp_id_col: str="cat_main",
    threshold: int = 80,
):
    """
    Perform WRatio name matching within spatial candidates.

    Returns
    -------
    GeoDataFrame with:
    - matched_id_name
    - name_score
    """

    # clean names for matching
    id_to_name_clean = compared_gdf.set_index(comp_id)[comp_name_col].apply(clean_name).apply(extract_prinmary_str).to_dict()
    # raw names for storage
    id_to_name_raw = compared_gdf.set_index(comp_id)[comp_name_col].to_dict()
    # raw compared df category
    id_to_cat = compared_gdf.set_index(comp_id)[comp_id_col].to_dict()

    matched_ids = []
    scores = []
    loc_dists = []
    matched_names = []
    matched_cats = []

    for _, row in reference_gdf.iterrows():
        query = extract_prinmary_str(clean_name(row.get(re_name_col)))

        if not isinstance(query, str) or not row["cand_ids"]:
            matched_ids.append(pd.NA)
            scores.append(pd.NA)
            loc_dists.append(pd.NA)
            matched_names.append(pd.NA)
            matched_cats.append(pd.NA)
            continue

        cand_names = [id_to_name_clean.get(cid, "") for cid in row["cand_ids"]]

        top_matches = process.extract(
            query,
            cand_names,
            scorer=fuzz.WRatio,
            limit=5
        )

        best_score = -1
        best_pos = None

        for name, wr, pos in top_matches:

            pr = fuzz.partial_ratio(query, name)
            ts = fuzz.token_sort_ratio(query, name)
            tset = fuzz.token_set_ratio(query, name)

            combined = max(wr, pr, ts, tset)

            if combined > best_score:
                best_score = combined
                best_pos = pos

        score = best_score

        if score >= threshold:
            matched_ids.append(row["cand_ids"][best_pos])
            scores.append(score)
            loc_dists.append(row["cand_dist_m"][best_pos])
            matched_names.append(id_to_name_raw.get(row["cand_ids"][best_pos]))
            matched_cats.append(id_to_cat.get(row["cand_ids"][best_pos]))
        else:
            matched_ids.append(pd.NA)
            scores.append(score)
            loc_dists.append(pd.NA)
            matched_names.append(pd.NA)
            matched_cats.append(pd.NA)


    result = reference_gdf.copy()
    result["matched_id"] = matched_ids
    result["name_score"] = scores
    result["location_distance"] = loc_dists
    result["matched_name"] = matched_names
    result["matched_cat_main"] = matched_cats

    return result

In [5]:
import pandas as pd
import geopandas as gpd
from rapidfuzz import fuzz
import re
import unicodedata

def clean_name(s):
    if not isinstance(s, str):
        return ""

    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c)) # 1. unicode normalize (remove accents)
    s = s.upper() # 2. uppercase
    s = re.sub(r"\([^)]*\)", "", s) 
    s = s.encode("ascii", errors="ignore").decode() # 4. remove emoji / non ascii
    s = re.sub(r"[^\w\s]", " ", s) # 5. replace special chars with space
    s = re.sub(r"\s+", " ", s) # 6. collapse spaces

    return s.strip()

def address_score_check(
    reference_gdf: gpd.GeoDataFrame,
    compared_gdf: gpd.GeoDataFrame,
    addr_col_ref: str = "addr_simple",
    addr_col_cmp: str = "address",
    matched_id_col: str = "matched_id",
    id_col: str = "id",
    out_col: str = "address_score",
    out_addr_col: str = "matched_address", 
):
    """
    Compute address similarity score (0-100) for already-matched rows.

    Only rows with non-null `matched_id_col` will be scored.
    Others will have NaN.

    Returns
    -------
    GeoDataFrame with a new column `out_col`.
    """

    # map: compared id -> compared address
    id_to_addr_clean = compared_gdf.set_index(id_col)[addr_col_cmp].apply(clean_name).to_dict()
    id_to_addr_raw = compared_gdf.set_index(id_col)[addr_col_cmp].to_dict()

    scores = []
    matched_addrs = []

    for _, row in reference_gdf.iterrows():
        matched_id = row.get(matched_id_col)

        # no matched id -> no score
        if pd.isna(matched_id):
            scores.append(pd.NA)
            matched_addrs.append(pd.NA)
            continue

        addr_ref = clean_name(row.get(addr_col_ref))
        addr_cmp = id_to_addr_clean.get(matched_id)

        if isinstance(addr_cmp, str) and addr_cmp.strip():
            matched_addrs.append(id_to_addr_raw.get(matched_id))
        else:
            matched_addrs.append(pd.NA)

        # missing address on either side -> no score
        if not isinstance(addr_ref, str) or not addr_ref.strip():
            scores.append(pd.NA)
            continue
        if not isinstance(addr_cmp, str) or not addr_cmp.strip():
            scores.append(pd.NA)
            continue

        wr = fuzz.WRatio(addr_ref, addr_cmp)
        pr = fuzz.partial_ratio(addr_ref, addr_cmp)
        ts = fuzz.token_sort_ratio(addr_ref, addr_cmp)
        tset = fuzz.token_set_ratio(addr_ref, addr_cmp)

        scores.append(int(max(wr, pr, ts, tset)))

    result = reference_gdf.copy()
    result[out_col] = scores
    result[out_addr_col] = matched_addrs
    return result

In [6]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

def calculate_similarity_check(
    df, 
    cat_col_ref: str = "primary_type", 
    cat_col_cmp: str = "matched_cat_main", 
    id_col: str = "matched_id", 
    result_col: str =  "category_sim",
):

    # 1. Setup Device
    device = "mps" if torch.backends.mps.is_available() else "cpu"
    model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
    
    # 2. Primary Gatekeeper: matched_id must be present
    mask = df[id_col].notna() & (df[id_col].astype(str).str.strip() != "")
    
    # 3. Create a helper to handle potential Nulls in text columns
    temp_df = df[mask].copy()
    
    # Identify where text is actually missing within the masked rows
    text_missing_mask = temp_df[cat_col_ref].isna() | temp_df[cat_col_cmp].isna()
    
    # Fill NaNs with empty strings just for the encoding step
    texts_1 = temp_df[cat_col_ref].fillna("").astype(str).tolist()
    texts_2 = temp_df[cat_col_cmp].fillna("").astype(str).tolist()

    print(f"Encoding {len(temp_df)} rows...")

    # 4. Batch Encoding
    emb1 = model.encode(texts_1, batch_size=256, show_progress_bar=True, convert_to_tensor=True)
    emb2 = model.encode(texts_2, batch_size=256, show_progress_bar=True, convert_to_tensor=True)

    # 5. Calculate Similarity
    scores = torch.nn.functional.cosine_similarity(emb1, emb2, dim=1).cpu().numpy()
    
    # 6. Apply NaN to rows where text was missing
    # Even if we encoded an empty string, the result isn't "real" if data was missing
    scores[text_missing_mask.values] = np.nan

    # 7. Map back to original dataframe
    df[result_col] = np.nan
    df.loc[mask, result_col] = scores
    
    return df

In [7]:
la_gplc = gpd.read_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/LA MSA/la_msa_poi/la_gplc.geojson')
la_gplc

,circle_id,id,name,address,primary_type,lat,lon,business_status,primary_cat,addr_simple,naics_code,naics_definition,geometry
0,22,ChIJ2VvvK4xy3YARFZspg5WuhrM,Electric Vehicle Charging Station,"410 Avalon Cyn Rd, Avalon, CA 90704, USA",electric_vehicle_charging_station,33.339858,-118.328966,OPERATIONAL,automotive,410 Avalon Cyn Rd,8111.0,Automotive Repair and Maintenance,POINT (-118.32897 33.33986)
1,23,ChIJX6QzILty3YARsnHmiRpR9Fw,Santa Catalina Island Gas Station,"Pebbly Beach Rd, Avalon, CA 90704, USA",gas_station,33.335206,-118.311799,OPERATIONAL,automotive,Pebbly Beach Rd,447.0,Gasoline Stations,POINT (-118.31180 33.33521)
2,23,ChIJX6QzILty3YARCbp1XWak4zg,Sonlight Auto,"Pebbly Beach Rd, Avalon, CA 90704, USA",car_repair,33.331990,-118.310062,OPERATIONAL,automotive,Pebbly Beach Rd,8111.0,Automotive Repair and Maintenance,POINT (-118.31006 33.33199)
3,23,ChIJE2xZ27ly3YAREjuKuQ75vfc,Electric Vehicle Charging Station,"1 Pebbly Beach Rd, Avalon, CA 90704, USA",electric_vehicle_charging_station,33.334863,-118.311751,OPERATIONAL,automotive,1 Pebbly Beach Rd,8111.0,Automotive Repair and Maintenance,POINT (-118.31175 33.33486)
4,28,ChIJ70LrNgBv3YARdKPC7ob-hbQ,Blackjack Shade Structure,"Avalon, CA 90704, USA",rest_stop,33.391372,-118.396269,OPERATIONAL,automotive,Avalon,7139.0,Other Amusement and Recreation Industries,POINT (-118.39627 33.39137)
...,...,...,...,...,...,...,...,...,...,...,...,...,...
39824,250,ChIJI3bzjr7v6YARIP1wV0XKtzk,Flying J/Frazier Mountain Park Rd. (760789),"California 93243, USA",bus_stop,34.817052,-118.886414,OPERATIONAL,transportation,California 93243,4851.0,Urban Transit Systems,POINT (-118.88641 34.81705)
39825,250,ChIJBzFz3S_v6YARWsF9y00pUsE,CAT Scale,"42810 Frazier Mountain Rd, Frazier Park, CA 93...",transportation_service,34.815689,-118.887895,OPERATIONAL,transportation,42810 Frazier Mountain Rd,4859.0,Other Transit and Ground Passenger Transportation,POINT (-118.88790 34.81569)
39826,250,ChIJe0tr67_v6YARTnbpF8rfPjo,Clinica Sierra Vista/Lebec Rd. (760766),"Lebec, CA 93243, USA",bus_stop,34.820765,-118.884328,OPERATIONAL,transportation,Lebec,4851.0,Urban Transit Systems,POINT (-118.88433 34.82077)
39827,250,ChIJOQgIEwTu6YARQ1GCBNmSyoE,Post Office/Monterey Trail (760830),"Frazier Park, CA 93225, USA",bus_station,34.820234,-118.945765,OPERATIONAL,transportation,Frazier Park,4851.0,Urban Transit Systems,POINT (-118.94577 34.82023)


In [8]:
la_fsq = gpd.read_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/LA MSA/la_msa_poi/la_fsq.geojson')
la_fsq

,fsq_place_id,name,latitude,longitude,address,locality,region,postcode,admin_region,post_town,...,fsq_category_ids,fsq_category_labels,placemaker_url,unresolved_flags,geom,bbox,cat_str,cat_main,cat_alt,geometry
0,5514e1e3498e2dd5c01df382,Spanish for Adults Online,32.980315,-118.557569,NaN,NaN,CA,90068,NaN,NaN,...,4bf58dd8d48988d13b941735,Community and Government > Education,https://foursquare.com/placemakers/review-plac...,NaN,AAAAAAHAXaOvN0OBREBAfXry9rZd,"{'xmin': '-118.55756932823448', 'ymin': '32.98...",Community and Government > Education,Community and Government,Education,POINT (-118.55757 32.98031)
1,52a0984311d2c0b1f972ce56,SCI Old PBX Site,32.979018,-118.553763,NaN,NaN,CA,NaN,NaN,NaN,...,4bf58dd8d48988d165941735,Landmarks and Outdoors > Scenic Lookout,https://foursquare.com/placemakers/review-plac...,NaN,AAAAAAHAXaNw28GGt0BAfVB1pgW8,"{'xmin': '-118.55376333140417', 'ymin': '32.97...",Landmarks and Outdoors > Scenic Lookout,Landmarks and Outdoors,Scenic Lookout,POINT (-118.55376 32.97902)
2,4ec3dbaeb63468c86d55f14d,The Commons,32.997933,-118.555278,NaN,Coronado,CA,NaN,NaN,NaN,...,4bf58dd8d48988d130941735,Landmarks and Outdoors > Structure,https://foursquare.com/placemakers/review-plac...,NaN,AAAAAAHAXaOJrGs890BAf7xG7TpE,"{'xmin': '-118.55527792427516', 'ymin': '32.99...",Landmarks and Outdoors > Structure,Landmarks and Outdoors,Structure,POINT (-118.55528 32.99793)
3,4cddd5eec4f6a35deff5c56c,San Clemente Island Commons,32.998234,-118.556404,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,https://foursquare.com/placemakers/review-plac...,NaN,AAAAAAHAXaOcH9dP0UBAf8YknRNN,"{'xmin': '-118.55640407587568', 'ymin': '32.99...",NaN,NaN,NaN,POINT (-118.55640 32.99823)
4,4ed314e2b634dd2994b4c7c6,Room 207,32.998383,-118.555649,NaN,NaN,NaN,NaN,NaN,NaN,...,4d954b06a243a5684965b473,Community and Government > Residential Buildin...,https://foursquare.com/placemakers/review-plac...,NaN,AAAAAAHAXaOPwAAAAEBAf8sAAAAA,"{'xmin': '-118.55564880371094', 'ymin': '32.99...",Community and Government > Residential Buildin...,Community and Government,Residential Building,POINT (-118.55565 32.99838)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1118178,65f83a4d11744101a4361fbc,Ram Steel Fabrication Inc,34.766547,-117.880867,NaN,Lancaster,CA,93535,NaN,NaN,...,63be6904847c3692a84b9b73,Business and Professional Services > Metals Su...,https://foursquare.com/placemakers/review-plac...,NaN,AAAAAAHAXXhgIH/h/0BBYh41FMe1,"{'xmin': '-117.88086712349467', 'ymin': '34.76...",Business and Professional Services > Metals Su...,Business and Professional Services,Metals Supplier,POINT (-117.88087 34.76655)
1118179,4f3d1779e4b085c2d961a9bc,South Gate Edwards AFB,34.787155,-117.916437,NaN,NaN,NaN,NaN,NaN,NaN,...,4bf58dd8d48988d12d941735,Landmarks and Outdoors > Monument,https://foursquare.com/placemakers/review-plac...,NaN,AAAAAAHAXXqm5lCP5EBBZMGBfvDE,"{'xmin': '-117.91643674723997', 'ymin': '34.78...",Landmarks and Outdoors > Monument,Landmarks and Outdoors,Monument,POINT (-117.91644 34.78716)
1118180,511bc1c8e4b02d59790cb4ab,AT&T cell site LA0206,34.760030,-117.799596,NaN,Lancaster,CA,93535,NaN,NaN,...,4bf58dd8d48988d130941735,Landmarks and Outdoors > Structure,https://foursquare.com/placemakers/review-plac...,NaN,AAAAAAHAXXMsk4VUV0BBYUitVdfl,"{'xmin': '-117.79959571857886', 'ymin': '34.76...",Landmarks and Outdoors > Structure,Landmarks and Outdoors,Structure,POINT (-117.79960 34.76003)
1118181,4e655bcae4cdf1e2c08c43a8,Rocketsite,34.817064,-117.825914,NaN,NaN,NaN,NaN,NaN,NaN,...,4bf58dd8d48988d15f941735,Landmarks and Outdoors > Field,https://foursquare.com/placemakers/review-plac...,NaN,AAAAAAHAXXTbxgzs2kBBaJWRasJB,"{'xmin': '-117.82591391813494', 'ymin': '34.81...",Landmarks and Outdoors > Field,Landmarks and Outdoors,Field,POINT (-117.82591 34.81706)


In [9]:
la_gplc_fsq = search_spatial_candidates(reference_gdf=la_gplc, compared_gdf=la_fsq, id_col = "fsq_place_id", k=100)

In [10]:
la_gplc_fsq = match_by_name(reference_gdf=la_gplc_fsq, compared_gdf=la_fsq, re_name_col = "name", comp_name_col = "name", comp_id = "fsq_place_id", comp_id_col ="cat_alt",  threshold=80)

In [11]:
la_gplc_fsq = address_score_check(reference_gdf=la_gplc_fsq, compared_gdf=la_fsq, addr_col_ref = "addr_simple", addr_col_cmp = "address", id_col = "fsq_place_id")

In [12]:
la_gplc_fsq = calculate_similarity_check(la_gplc_fsq, cat_col_ref = "naics_definition", cat_col_cmp = "matched_cat_main", id_col = "matched_id", result_col =  "category_sim")

Encoding 33015 rows...


Batches:   0%|          | 0/129 [00:00<?, ?it/s]

Batches:   0%|          | 0/129 [00:00<?, ?it/s]

In [13]:
# transfer the X and Y into float type and deal with the address score
cols_to_fix = ['name_score', 'location_distance', 'address_score', 'category_sim']
for col in cols_to_fix:
    la_gplc_fsq[col] = pd.to_numeric(la_gplc_fsq[col], errors='coerce')

In [ ]:
df = la_gplc_fsq[la_gplc_fsq["matched_id"].notna()].copy()

N = 1000

weights = df["primary_cat"].map(
    df["primary_cat"].value_counts(normalize=True)
)

df_sample = df.sample(
    n=N,
    weights=weights,
    random_state=42
)
df_sample[['id','name','addr_simple','naics_definition', 'matched_id','matched_name','matched_address','matched_cat_main','location_distance']].to_csv('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/bspo_gplc_fsq_2000_sample.csv', index=False)

In [14]:
df_sample_check = pd.read_csv('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/LA MSA/la_google_comparison/la_gplc_fsq_1000_sample.csv')
df_sample_check = df_sample_check.drop(columns='location_distance')
df_sample_check = df_sample_check.merge(la_gplc_fsq[['id','name_score','location_distance','address_score','category_sim']], left_on="id", right_on="id", how="left")

In [15]:
df_sample_check['is_true'].value_counts()

is_true
1    935
0     65
Name: count, dtype: int64

In [16]:
df = df_sample_check.copy()
from sklearn.preprocessing import StandardScaler

X = df[['name_score', 'location_distance', 'address_score', 'category_sim']]
y = df['is_true']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.25,
    random_state=42,
    stratify=y  # keep the same proportion of True vs False in training set and test set
)

In [18]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

xgb_clf = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    enable_categorical=False,
    eval_metric='auc',
    random_state=42
)

xgb_clf.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False  
)

xgb_y_pred = xgb_clf.predict(X_test)
xgb_y_prob = xgb_clf.predict_proba(X_test)[:, 1]

xgb_cr = classification_report(y_test, xgb_y_pred)
xgb_auc = roc_auc_score(y_test, xgb_y_prob)

print(xgb_cr)
print("XGBoost AUC:", xgb_auc)

              precision    recall  f1-score   support

           0       0.69      0.56      0.62        16
           1       0.97      0.98      0.98       234

    accuracy                           0.96       250
   macro avg       0.83      0.77      0.80       250
weighted avg       0.95      0.96      0.95       250

XGBoost AUC: 0.906784188034188


In [19]:
mask = la_gplc_fsq['matched_id'].notnull()
df_pred = la_gplc_fsq.loc[mask].copy()

feature_cols = ['name_score', 'location_distance', 'address_score', 'category_sim']

X_new = scaler.transform(df_pred[feature_cols])
df_pred['true_match_prob'] = xgb_clf.predict_proba(X_new)[:, 1]
df_pred['is_true_match'] = df_pred['true_match_prob'] >= 0.5

la_gplc_fsq.loc[mask, 'is_true_match'] = df_pred['is_true_match']
la_gplc_fsq.loc[mask, 'true_match_prob'] = df_pred['true_match_prob']

In [20]:
la_gplc_fsq['is_true_match'].value_counts()

is_true_match
True     29363
False     3652
Name: count, dtype: int64

In [21]:
la_gplc_fsq.drop(columns=['cand_ids','cand_dist_m']).to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/LA MSA/la_google_comparison/la_gplc_fsq.geojson')